# Team STAR - Step 3

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tsim

Here we define a circuit that corresponds to a possible implementation of the problem.

We use two 3-distance surface codes, one for the main logical qubit, another for the ancilary one.

In [ ]:
# Main Logical Qubit: Data (0-8), Ancilla (9-16)
# Ancillary Logical Qubit: Data (100-108), Ancilla (109-116)

star_circuit = """
# 1. INITIALIZATION

# Main Logical Qubit
QUBIT_COORDS(0.5, 0.5) 0
QUBIT_COORDS(1.5, 0.5) 1
QUBIT_COORDS(2.5, 0.5) 2
QUBIT_COORDS(0.5, 1.5) 3
QUBIT_COORDS(1.5, 1.5) 4
QUBIT_COORDS(2.5, 1.5) 5
QUBIT_COORDS(0.5, 2.5) 6
QUBIT_COORDS(1.5, 2.5) 7
QUBIT_COORDS(2.5, 2.5) 8
QUBIT_COORDS(1, 0) 9
QUBIT_COORDS(2, 1) 10
QUBIT_COORDS(1, 2) 11
QUBIT_COORDS(2, 3) 12
QUBIT_COORDS(1, 1) 13
QUBIT_COORDS(3, 1) 14
QUBIT_COORDS(0, 2) 15
QUBIT_COORDS(2, 2) 16

# Ancillary Logical Qubit
QUBIT_COORDS(4, 0.5) 100
QUBIT_COORDS(5, 0.5) 101
QUBIT_COORDS(6, 0.5) 102
QUBIT_COORDS(4, 1.5) 103
QUBIT_COORDS(5, 1.5) 104
QUBIT_COORDS(6, 1.5) 105
QUBIT_COORDS(4, 2.5) 106
QUBIT_COORDS(5, 2.5) 107
QUBIT_COORDS(6, 2.5) 108
QUBIT_COORDS(4.5, 0) 109
QUBIT_COORDS(5.5, 1) 110
QUBIT_COORDS(4.5, 2) 111
QUBIT_COORDS(5.5, 3) 112
QUBIT_COORDS(4.5, 1) 113
QUBIT_COORDS(6.5, 1) 114
QUBIT_COORDS(3.5, 2) 115
QUBIT_COORDS(5.5, 2) 116

# Reset all physical qubits to |0>
R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
R 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116

TICK

# 2. STATE INJECTION (Small-angle R_Z on Ancilla)
# We prepare the Ancillary logical qubit in a rotated state transversally
H 100 101 102 103 104 105 106 107 108
R_Z(0.156) 100 101 102 103 104 105 106 107 108

TICK

# 3. TRANSVERSAL LOGICAL CNOT (Main -> Ancilla)
# Each physical data qubit in Main controls the corresponding one in Ancilla
CNOT 0 100
CNOT 1 101
CNOT 2 102
CNOT 3 103
CNOT 4 104
CNOT 5 105
CNOT 6 106
CNOT 7 107
CNOT 8 108

TICK

# 4. SYNDROME EXTRACTION (Example: One round for both)
# (In a full sim, we would repeat this for multiple rounds)
# Example: Z-Stabilizer on Main (Data 0,1,3,4) using Ancilla 9
H 9
CNOT 0 9
CNOT 1 9
CNOT 3 9
CNOT 4 9
H 9
M 9
DETECTOR(0, 0, 0) rec[-1]

TICK

# 5. POST-SELECTION MEASUREMENT
# Measure Ancilla Logical Qubit in X-basis for teleportation
H 100 101 102 103 104 105 106 107 108
M 100 101 102 103 104 105 106 107 108

TICK

# The logical X result is the XOR of these physical measurements
# We use a detector to filter for the '0' outcome (identity)
OBSERVABLE_INCLUDE(0) rec[-1] rec[-2] rec[-3] rec[-4] rec[-5] rec[-6] rec[-7] rec[-8] rec[-9]
"""

# Compile for GPU
circuit = tsim.Circuit(star_circuit)

Visualising time slices of the circuit of the steps followed in the circuit

In [86]:
circuit.diagram("timeslice-svg", height=400, width=1000)

To implement we would need to define the decoding/post-selection logic in Python after running sampler.sample(), similar to what we did in Step 2, but this time doing it for 18 ancillary qubits in total, since we have two logical qubits.